In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *
from prompts.synonym_context_prompt import *

load_dotenv()

pd.set_option('display.max_rows', None)
#"gpt-4.1-mini"
#"gpt-4o-mini"
#"gpt-4.1"
#"gpt-4o"

model_ = "gpt-5.1"
api_key = os.getenv("API_KEY")


llm = llm_call(model_version = model_, api_key= api_key)

In [2]:
# "Credit-TEST-POLLUTED.NORND-activity-0.3-0"
# "Credit-TEST-SYNONYM-0.3-0"
# "Credit-TRAIN-DISTORTED-activity-0.3-0"
#Pub-Collateral
#Credit-TRAIN-HOMONYM-0.3-0
#Credit-TEST-CLEAN
test = "Credit-TEST-POLLUTED.NORND-activity-0.3-0"
LOG_NAME = f"./dataset/{test}.csv" 

df_new, cases_json = build_event_jsons(log_name = LOG_NAME, chunk_cases = 10)

#sample_size = int(len(df_new['case_id'].unique()) * 0.25)
#np.random.seed(42)
#selected_case_ids = np.random.choice(df_new['case_id'].unique(), size=sample_size, replace=False)
#df_new = df_new[df_new['case_id'].isin(selected_case_ids)].copy()


df_new.head(3)


,event_id,case_id,activity,timestamp,label
0,0,15,Check for completeness,2023-09-29 16:58:01.944,NaN
1,1,15,New online application received_nan,2023-09-29 16:58:01.944,polluted Label(Activity)
2,2,15,Perform checks_Clerk-000001,2023-09-29 17:14:52.875,polluted Label(Activity)


In [3]:
activity_list = df_new['activity'].unique().tolist()
activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
print(activity_list_json)

[
    "Check for completeness",
    "New online application received_nan",
    "Perform checks_Clerk-000001",
    "Make decision",
    "notify reject",
    "time out",
    "EVENT 13 END_nan",
    "Check for completeness_Clerk-000001",
    "Request info",
    "info received_nan",
    "Check for completeness_Clerk-000003",
    "Perform checks",
    "review request received_nan",
    "Perform checks_Clerk-000005",
    "Notify accept_Manager-000004",
    "Deliver card",
    "New online application received",
    "info received",
    "Request info_Manager-000003",
    "Check for completeness_Clerk-000005",
    "EVENT 13 END",
    "Perform checks_Clerk-000003",
    "review request received",
    "Make decision_Manager-000005",
    "Notify accept",
    "Deliver card_Manager-000001",
    "Request info_Manager-000005",
    "Perform checks_Clerk-000004",
    "Request info_Manager-000006",
    "notify reject_Manager-000003",
    "Check for completeness_Clerk-000004",
    "Notify accept_Manager-00

In [4]:
SYSTEM_PROMPT_STEP1 = """
You are an expert Process Mining Data Pre-processor.
Your goal is to filter a raw list of activity names based on specific criteria provided in the User Prompt.

### KNOWLEDGE BASE: IMPERFECTION PATTERNS
Use these definitions to identify which labels belong to which category.

1.  **Polluted Labels (Mutable Qualifiers):**
    Labels that share a immutable boiler-plate text but differ due to mutable text (e.g., embedded IDs or codes).
    * **Detection Criteria:**
        * **Long Numeric IDs:** 8+ digits (e.g., `20260122`, `9988776655`).
        * **Mixed Codes:** 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
        * **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

2.  **Distorted Labels (Character-Level Corruption):**
    Labels containing specific character-level corruptions (typos, OCR faults) of a canonical form. Unlike synonyms, these are "Noise".
    * **Detection Criteria:**
        1.  **Case Mutation:** Identical spelling, different capitalization (e.g., "Open" vs "open" vs "OPEN").
        2.  **Character Omission:** Exactly ONE missing character (e.g., "Invoce" vs "Invoice").
        3.  **Character Insertion:** Exactly ONE extra character (e.g., "Innvoice" vs "Invoice").
        4.  **Character Transposition:** Two adjacent characters swapped (e.g., "Ivnoice" vs "Invoice").
        5.  **Keyboard Proximity:** Exactly ONE character substituted by a QWERTY neighbor (e.g., "Invoicr" vs "Invoice").

3. **Synonymous Labels (Semantic Equivalence):**
   Labels that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step. 
   * **Detection Criteria (Ontology Rules):**
        1. **Linguistic & Domain Synonyms:** Different words representing the same concept within the process context (e.g., "Ship Item" vs "Dispatch Goods", "DrSeen" vs "Medical Assign").
        2. **Phrase Variation (Verb/Object Shift):** Labels sharing a core component (usually the Object) while using synonymous verbs or adjectives (e.g., "Create Invoice" vs "Generate Invoice", "Start instance" vs "Start process").
        3. **Grammatical Transformation:** Changing parts of speech (Noun ↔ Verb) or sentence structure while retaining the core meaning (e.g., "Give approval" vs "Approve", "Conduct analysis" vs "Analyze").
        4. **Containment & Refinement:** One label is a concise or verbose version of the other, often omitting non-essential adjectives, prepositions, or 'online/offline' qualifiers (e.g., "Receive signed contract" vs "Receive contract", "Register for course" vs "Register course").

### GLOBAL INSTRUCTION
- **Role:** Function as a logic engine. Do not assume all imperfections exist.
- **Priority:** The strict filtering logic in the **User Prompt** overrides general definitions here.
- **OUTPUT FORMAT:** Always return valid JSON as requested by the User Prompt.

"""

USER_PROMPT_DISTORTED_STEP1= f"""
### TASK: Filter Data for 'Distorted Labels' Pairs

**OBJECTIVE:**
Analyze the provided **INPUT DATA(activity list)** and extract labels related to **Distorted Labels (Character-Level Corruption)**.
You must **identify and collect** the following components for the output data:
1. All **Distorted Labels** found (even if their clean version is missing).
2. Any **Clean Labels** found **ONLY IF** they correspond to a Distorted Label present in the list.

**STRICT FILTERING LOGIC:**
1. **Identify Distortion Pairs:** Look for labels that are character-level variations of each other based on the System Prompt's 'Distorted Labels' criteria (Case Mutation, Omission, Insertion, Transposition, Keyboard Proximity).
2. **KEEP Related Labels:** If Label A is a distorted version of Label B (or vice versa), **KEEP BOTH A and B**.
3. **DISCARD Isolated Clean Labels:** If a label is "Clean" but has **NO** distorted variants in the provided list, **REMOVE IT**. (e.g., If 'Start' exists but 'Strt' does not, remove 'Start').
4. **DISCARD Polluted/Synonymous:** Remove labels that are purely 'Polluted Labels' or 'Synonymous Labels' if they are not part of a 'Distorted Labels' pattern.

**EDGE CASE HANDLING:**
- If NO Distorted labels are found (i.e., all labels are either perfectly unique clean labels, polluted, or distinct synonyms):
    - Return strictly `[]`.
- **Finding NOTHING is a valid result.** Do not include isolated clean labels just to populate the list.

**INPUT DATA:**
{activity_list_json}

***OUTPUT FORMAT GUIDELINES (PERFORMANCE OPTIMIZED)***
Return a JSON Object with two keys:
1. "found": Boolean (true if distorted labels exist, false otherwise).
2. "data": List of strings.

**Example (Found):**
{{ "found": true, "data": ["Check", "Chekc"] }}

**Example (Not Found - SPEED PRIORITY):**
{{ "found": false, "data": [] }}

**CONSTRAINT:**
- Determine the "found" value FIRST. If false, output `[]` for data immediately.
- Output ONLY the JSON.
"""
USER_PROMPT_POLLUTED_STEP1 = f"""
### TASK: Filter Data for 'Polluted Label' Candidates

**OBJECTIVE:**
Analyze the provided **INPUT DATA(activity list)** and extract labels exhibiting **Polluted Labels (Mutable Qualifiers)**.
You must **identify and collect** the following components for the output data:
1. All **Polluted Labels** found (labels with embedded IDs, dates, or complex codes).
2. Any **Immutable Boiler-plate Text (Clean Label)** found **ONLY IF** they correspond to a Polluted Label present in the list.

**STRICT FILTERING LOGIC:**
1. **Identify Polluted Labels:** Look for labels containing variable identifiers based on the System Prompt's criteria (IDs, Codes) and **KEEP** them regardless of whether their template base exists (e.g., Keep `["Step_X99"]` even if `"Step"` is missing).
2. **KEEP Clean/Immutable Boiler-plate Text Labels :** Check for the "Clean" version of any detected Polluted Label and **KEEP** it **ONLY IF** it corresponds to a Polluted Label present in the list (e.g., Keep `"Step"` only if `"Step_X99"` exists).
3. **DISCARD Isolated Clean Labels:** If a label is "Clean" but has **NO** polluted variants in the provided list, **REMOVE IT**. (e.g., Remove `"Start Process"` if no `"Start Process_123"` exists).
4. **DISCARD Distorted/Synonymous:** Remove labels that are purely 'Distorted Labels' or 'Synonymous Labels' if they are not part of a 'Polluted Labels' pattern.

**EDGE CASE HANDLING:**
- If NO Polluted labels are found (i.e., all labels are clean, distorted, or synonyms):
    - Return strictly `[]` (with "found": false).
- **Finding NOTHING is a valid result.** Do not include isolated clean labels just to populate the list.

**INPUT DATA:**
{activity_list_json}

***OUTPUT FORMAT GUIDELINES (PERFORMANCE OPTIMIZED)***
Return a JSON Object with two keys:
1. "found": Boolean (true if polluted labels exist, false otherwise).
2. "data": List of strings.

**Example (Found):**
{{ "found": true, "data": ["Case_20240101", "Case_20240102", "Case"] }}

**Example (Not Found - SPEED PRIORITY):**
{{ "found": false, "data": [] }}

**CONSTRAINT:**
- Determine the "found" value FIRST. If false, output `[]` for data immediately.
- Output ONLY the JSON.
"""
USER_PROMPT_SYNONYMOUS_STEP1 = f"""
### TASK: Filter Data for 'Synonymous Labels' Candidates

**OBJECTIVE:**
Analyze the provided **INPUT DATA(activity list)** and extract labels related to **Synonymous Labels (Semantic Equivalence)**.
You must **identify and collect** the following components for the output data:
1. All labels that form a **Synonym Group** (two or more labels representing the same process step despite syntactic differences).

**STRICT FILTERING LOGIC:**
1. **Identify Synonym Pairs:** Look for different words or phrases that share the same semantic meaning based on the System Prompt's criteria (Linguistic Synonyms, Phrase Variation, Grammatical Transformation, Containment).
2. **KEEP Related Labels:** If Label A is a semantic synonym of Label B, **KEEP BOTH A and B**. (Unlike Distorted/Polluted, typically all members of a synonym group are valid words, so keep the entire group).
3. **DISCARD Isolated Labels:** If a label is valid but has **NO** semantic synonyms in the provided list, **REMOVE IT**. (e.g., If 'Archive' exists but no synonyms like 'Store' or 'Save' exist, remove 'Archive').
4. **DISCARD Distorted/Polluted:** Remove labels that are purely 'Distorted Labels' (typos) or 'Polluted Labels' (IDs) if they are not part of a 'Synonymous Labels' pattern.

**EDGE CASE HANDLING:**
- If NO Synonymous pairs are found (i.e., all labels are unique/isolated, distorted, or polluted):
    - Return strictly `[]` (with "found": false).
- **Finding NOTHING is a valid result.** Do not force-fit vaguely similar words; strict semantic equivalence is required.

**INPUT DATA:**
{activity_list_json}

***OUTPUT FORMAT GUIDELINES (PERFORMANCE OPTIMIZED)***
Return a JSON Object with two keys:
1. "found": Boolean (true if synonymous labels exist, false otherwise).
2. "data": List of strings.

**Example (Found):**
{{ "found": true, "data": ["Create Invoice", "Generate Invoice", "Make Bill"] }}

**Example (Not Found - SPEED PRIORITY):**
{{ "found": false, "data": [] }}

**CONSTRAINT:**
- Determine the "found" value FIRST. If false, output `[]` for data immediately.
- Output ONLY the JSON.
"""

In [5]:
prompt = [{"role": "system", "content": SYSTEM_PROMPT_STEP1},
          {"role":"user","content": USER_PROMPT_POLLUTED_STEP1}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
synonym_list_json = json.dumps(test_output['data'], indent=4, ensure_ascii=False)


In [6]:
for k, v in test_output.items():
    print(f"{k}:")
    if isinstance(v, list):  # 값이 리스트인 경우
        if v:  # 리스트가 비어 있지 않을 때
            print(*v, sep="\n")
        else:  # 리스트가 비어 있을 때
            print(" (empty list)")
    else:  # 'found'와 같이 리스트가 아닌 경우
        print(f" {v}")
    print() # 가독성을 위한 한 줄 띄우기

found:
 True

data:
New online application received_nan
Perform checks_Clerk-000001
EVENT 13 END_nan
Check for completeness_Clerk-000001
info received_nan
Check for completeness_Clerk-000003
review request received_nan
Perform checks_Clerk-000005
Notify accept_Manager-000004
Request info_Manager-000003
Check for completeness_Clerk-000005
Perform checks_Clerk-000003
Make decision_Manager-000005
Deliver card_Manager-000001
Request info_Manager-000005
Perform checks_Clerk-000004
Request info_Manager-000006
notify reject_Manager-000003
Check for completeness_Clerk-000004
Notify accept_Manager-000001
Check for completeness_Clerk-000006
Check for completeness_Clerk-000002
Request info_Manager-000002
Make decision_Manager-000001
Notify accept_Manager-000006
Deliver card_Manager-000004
Make decision_Manager-000002
notify reject_Manager-000005
notify reject_Manager-000006
Request info_Manager-000004
Make decision_Manager-000006
Deliver card_Manager-000006
Perform checks_Clerk-000002
time out_na

In [7]:
import pm4py

def get_synonym_context(df: pd.DataFrame,
                        case_col: str = 'case_id',
                        time_col: str = 'timestamp',
                        act_col: str = 'activity',
                        filter_list: set = None):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    dfg, start_activities, end_activities = pm4py.discover_dfg(df_pm4py)
    def get_activity_context(activity, dfg_dict):
        predecessors = {k[0]: v for k, v in dfg_dict.items() if k[1] == activity}
        successors = {k[1]: v for k, v in dfg_dict.items() if k[0] == activity}
        total_pred = sum(predecessors.values())
        total_succ = sum(successors.values())
        def format_to_list(dist_dict, total):
            if total == 0: return []
            items = [(k, v/total) for k, v in dist_dict.items() if (v/total) >= 0.05]
            items.sort(key=lambda x: x[1], reverse=True)
            return [k for k, v in items]
        return format_to_list(predecessors, total_pred), format_to_list(successors, total_succ)
    all_activities = sorted(df[act_col].unique())
    flow_data_list = []
    for act in all_activities:
        if filter_list is not None and act not in filter_list:
            continue
        pred, succ = get_activity_context(act, dfg)
        flow_data_list.append({
            'activity': act,
            'predecessors': pred, # 이제 리스트입니다 ['A', 'B']
            'successors': succ    # 이제 리스트입니다 ['C', 'D']
        })
        
    json_flow_context = json.dumps(flow_data_list, indent=2, ensure_ascii=False)
    return json_flow_context

    
##print(synonym_clusters_json)

target_activities_set = test_output['data']
synonym_context_json = get_synonym_context(
    df=df_new,          
    filter_list=target_activities_set 
)
print(synonym_context_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": [
      "info received",
      "info received_nan",
      "review request received",
      "review request received_nan"
    ],
    "successors": [
      "New online application received",
      "Request info",
      "Perform checks",
      "New online application received_nan"
    ]
  },
  {
    "activity": "Check for completeness_Clerk-000001",
    "predecessors": [
      "info received",
      "info received_nan",
      "review request received",
      "review request received_nan"
    ],
    "successors": [
      "New online application received",
      "Perform checks",
      "Request info",
      "New online application received_nan"
    ]
  },
  {
    "activity": "Check for completeness_Clerk-000002",
    "predecessors": [
      "info received",
      "info received_nan",
      "review request received",
      "review request received_nan"
    ],
    "successors": [
      "New online application received",
     

In [8]:
SYSTEM_PROMPT_POLLUTED_STEP2  = """
You are an expert Process Mining Analyst.
Your goal is to summarize lists of activity labels into a single, descriptive **Process Stage Name**.

### CORE TASK
You will be given an activity and its lists of **Predecessors** (incoming flow) and **Successors** (outgoing flow).
You must analyze the labels in each list and determine the **Common Business Phase** they represent.

### SUMMARIZATION LOGIC (ABSTRACTION)
1. **Identify the Core Action:** Look at the verbs and objects in the list.
2. **Ignore Noise:** Disregard synonyms, typos, and minor variations.
3. **Formulate a Summary:** Create a short, natural language phrase that encapsulates the collective meaning.

### EXAMPLES (Demonstration Only)
- **Input List:** `["Wrap package", "Box items", "Pack goods", "Containerize"]`
- **Output Summary:** "Packaging Phase"

- **Input List:** `["MRI Scan", "X-Ray taken", "Blood test results"]`
- **Output Summary:** "Medical Diagnosis Stage"

- **Input List:** `["Ticket Resolved", "Issue Fixed", "Close Ticket", "Problem Solved"]`
- **Output Summary:** "Ticket Resolution"

### GLOBAL INSTRUCTION
- **Input:** JSON object with `activity`, `predecessors` (list), and `successors` (list).
- **Output:** JSON object where `predecessors` and `successors` are converted to **Strings** (Summaries).
"""

USER_PROMPT_POLLUTED_STEP2 = f"""
### TASK: Summarize Contextual Flow Lists

**OBJECTIVE:**
Analyze the **INPUT DATA**. Replace the list of strings in `predecessors` and `successors` with a **Single Summarized String** describing that process stage.

**STRICT EXECUTION STEPS:**
1. **Iterate** through every activity in the input.
2. **Analyze Predecessors:**
   - Read the list of predecessor labels.
   - Abstract their common meaning into one short phrase (e.g., "Quality Check Phase").
   - **Replace** the list with this string.
3. **Analyze Successors:**
   - Read the list of successor labels.
   - Abstract their common meaning into one short phrase.
   - **Replace** the list with this string.

**INPUT DATA:**
{synonym_context_json}

***OUTPUT FORMAT GUIDELINES***
Return a JSON Object with a single key `"summarized_context"`.
The value must be a list of objects where `predecessors` and `successors` are **STRINGS**, not lists.

**Example Output (Mental Model):**
{{
  "summarized_context": [
    {{
      "activity": "Ship Item",
      "predecessors": "Packaging Phase",     // Was ["Box items", "Wrap package"...]
      "successors": "Delivery Initiation"    // Was ["Truck loaded", "Dispatch"...]
    }},
    {{
      "activity": "Handle Error",
      "predecessors": "System Failure",      // Was ["Crash", "Server Down"...]
      "successors": "Recovery Process"       // Was ["Reboot", "Restart"...]
    }}
  ]
}}

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_POLLUTED_STEP2},
          {"role":"user","content": USER_PROMPT_POLLUTED_STEP2}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output_polluted_step2 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
step2_out_json = json.dumps(test_output_polluted_step2["summarized_context"], indent=2, ensure_ascii=False)
print(step2_out_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": "Customer information or review request received",
    "successors": "Online application registration and follow-up checks or info request"
  },
  {
    "activity": "Check for completeness_Clerk-000001",
    "predecessors": "Customer information or review request received",
    "successors": "Online application registration and follow-up checks or info request"
  },
  {
    "activity": "Check for completeness_Clerk-000002",
    "predecessors": "Customer information or review request received",
    "successors": "Online application registration and follow-up checks or info request"
  },
  {
    "activity": "Check for completeness_Clerk-000003",
    "predecessors": "Customer information or review request received",
    "successors": "Online application registration and follow-up checks or info request"
  },
  {
    "activity": "Check for completeness_Clerk-000004",
    "predecessors": "Customer information or review requ

In [9]:
SYSTEM_PROMPT_POLLUTED_STEP3 = """
You are an expert Process Mining Data Cleaner specializing in **Polluted Label Detection**.
Your goal is to identify the "Clean Label" (Canonical Form) and map all its "Polluted Variants" based on Context and Text Patterns.

### INPUT DATA
You will receive objects with:
1. `activity`: The label.
2. `predecessors`: A summarized string (Input Context).
3. `successors`: A summarized string (Output Context).

### KNOWLEDGE BASE: POLLUTED LABELS (Mutable Qualifiers)
A label is "Polluted" if it consists of a **Clean Root** followed by mutable text (Noise) like IDs or codes.
- **Detection Criteria:**
    - **Pattern:** `[Clean Label] + [Delimiter] + [ID/Code]`
    - **Delimiters:** `_`, `-`, `:`, `/`, `#`, `.`, or space.
    - **Noise Examples:** Long numeric IDs (8+ digits), Mixed Codes (e.g., `XJ9281`), User IDs (e.g., `Clerk-001`).

### DETECTION LOGIC: FUZZY CONTEXT & PATTERN
To map a Polluted Variant to a Clean Label, BOTH conditions must be met:

**CONDITION 1: CONTEXT SIMILARITY (Validation)**
- Compare `predecessors_A` vs `predecessors_B` AND `successors_A` vs `successors_B`.
- **Do not look for exact string matches.**
- **Rule:** The Clean Label and its Polluted Variant must share the **Same Process Context**.
    - *Reasoning:* If "Check_01" and "Check_02" are the same step, they must happen at the same point in the process.

**CONDITION 2: TEXTUAL CONTAINMENT (Root Check)**
- **Rule:** The "Clean Label" must be a substring or the root phrase of the "Polluted Variant".
- **LOGIC:** Within a contextually similar group, the **Shortest / Simplest** string is usually the Clean Label.

### GLOBAL INSTRUCTION
- **Output:** A JSON object where **Key** = Clean Label, **Value** = List of Polluted Variants.
- **Constraint:** Only include pairs where actual pollution is detected. Do not output clean labels that have no variants.
"""

USER_PROMPT_POLLUTED_STEP3 = f"""
### TASK: Clean vs. Polluted Mapping

**OBJECTIVE:**
Analyze the **INPUT DATA**. Identify "Clean Labels" and group their "Polluted Variants" based on the System Prompt's criteria.
**Key Instruction:** Be flexible with context descriptions. Focus on the **Core Meaning**.

**STRICT EXECUTION STEPS:**

1.  **Group by Context:**
    - Look at activities that share **Semantically Similar Predecessors AND Successors**.
    - (Use the "Fuzzy Context" logic: e.g., "Info Received" ≈ "Receipt of Info").

2.  **Identify Clean Root:**
    - Inside each context group, find the label that serves as the **Clean Root**.
    - *Hint:* It is usually the shortest string without numbers or special codes (e.g., "Check" vs "Check_01").

3.  **Map Variants:**
    - Identify other labels in the group that follow the pattern `Clean Root + Delimiter + Code`.
    - Verify they match the **Polluted Definition** (IDs, Mixed Codes).

4.  **Construct Output:**
    - Create a map: `{{ "Clean Label": ["Polluted_Var_1", "Polluted_Var_2"] }}`.

**INPUT DATA (Summarized Context):**
{step2_out_json}

***OUTPUT FORMAT GUIDELINES***
Return a strict JSON Object. Keys are strings, Values are lists of strings.

**Example Logic (Mental Model):**
- **Data:**
    * A: "Approve" (Context: X)
    * B: "Approve_Manager1" (Context: X)
    * C: "Approve_Manager2" (Context: X)
- **Analysis:**
    * Context Match: All share Context X.
    * Root Check: "Approve" is the shortest root. B and C contain "Approve" + "_" + ID.
- **Result:** `{{ "Approve": ["Approve_Manager1", "Approve_Manager2"] }}`

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_POLLUTED_STEP3},
          {"role":"user","content": USER_PROMPT_POLLUTED_STEP3}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output_polluted_step3 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
step3_out_json = json.dumps(test_output_polluted_step3, indent=4, ensure_ascii=False)


In [10]:
print("\n--- [Process Abstraction Report] ---\n")

for clean, vars in test_output_polluted_step3.items():
    print(f"Original: '{clean}' \n-> {vars})\n")



--- [Process Abstraction Report] ---

Original: 'Check for completeness' 
-> ['Check for completeness_Clerk-000001', 'Check for completeness_Clerk-000002', 'Check for completeness_Clerk-000003', 'Check for completeness_Clerk-000004', 'Check for completeness_Clerk-000005', 'Check for completeness_Clerk-000006'])

Original: 'Deliver card' 
-> ['Deliver card_Manager-000001', 'Deliver card_Manager-000002', 'Deliver card_Manager-000003', 'Deliver card_Manager-000004', 'Deliver card_Manager-000005', 'Deliver card_Manager-000006'])

Original: 'EVENT 13 END' 
-> ['EVENT 13 END_nan'])

Original: 'Make decision' 
-> ['Make decision_Manager-000001', 'Make decision_Manager-000002', 'Make decision_Manager-000003', 'Make decision_Manager-000004', 'Make decision_Manager-000005', 'Make decision_Manager-000006'])

Original: 'New online application received' 
-> ['New online application received_nan'])

Original: 'Notify accept' 
-> ['Notify accept_Manager-000001', 'Notify accept_Manager-000002', 'Noti

In [12]:
SYSTEM_PROMPT_POLLUTED_STEP4 = """
You are a strict Data Filtering Engine.
Your task is to extract event_ids based ONLY on exact string matching against a provided Reference Mapping.

### STRICT MATCHING RULES
1. **Reference Check Only:** The `activity` string in the log must match a string in the **Polluted Variant Lists (Values)** of the Reference Mapping exactly.
2. **No Pattern Recognition:** Do NOT try to identify "Polluted Labels" using heuristics (e.g., detecting numbers, IDs, dates, or codes). If a label is not explicitly in the provided list, IGNORE it.
3. **Key Exclusion:** If the `activity` matches the **Key** (Clean Label), IGNORE it.

### OUTPUT FORMAT
Return strictly a valid JSON object containing a single key "event_id" with a list of strings.
No markdown, no explanations.

**Example Structure:**
{
  "event_id": ["id_1", "id_2"]
}
"""
def get_polluted_user_prompt_step4(polluted_mapping, event_log_chunk):
    return f"""
### TASK: Strict Log Filtering (Polluted Variants)

**OBJECTIVE:**
Scan the **Target Event Log**. Identify `event_id`s where the `activity` exists in the **Values** of the Reference Mapping.

**1. REFERENCE MAPPING (Clean Key -> [Polluted Variants]):**
{polluted_mapping}

**2. TARGET EVENT LOG:**
{event_log_chunk}

***INSTRUCTIONS***
- Scan the `activity` field in the Event Log.
- Check if it exists in the **Values (Lists)** of the Reference Mapping.
- **DO NOT** use pattern matching or inference. Rely ONLY on the provided list.
- If found in the list, keep the `event_id`.

***OUTPUT***
Return ONLY the JSON object with the list of matching event IDs.
"""
target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)
all_predicted_ids = set() 
for case in target_cases:
    try:
        USER_PROMPT = get_polluted_user_prompt_step4(step3_out_json, case)
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT_POLLUTED_STEP4},
            {"role": "user", "content": USER_PROMPT}
        ]
        test_output = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
        if isinstance(test_output, dict):
            current_ids = test_output.get('event_id', [])
            all_predicted_ids.update(str(x) for x in current_ids)
            
    except Exception as e:
        print(f"Error processing a case: {e}")
        continue
target_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)]
ground_truth_df = target_df[target_df['label'].notna()]
actual_ids = set(ground_truth_df['event_id'].astype(str))
intersection = all_predicted_ids.intersection(actual_ids)
tp = len(intersection)
n_actual = len(actual_ids) 
n_pred = len(all_predicted_ids) 
recall = (tp / n_actual * 100) if n_actual > 0 else 0.0
precision = (tp / n_pred * 100) if n_pred > 0 else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print("\n" + "=" * 40)
print(f"   BATCH EVALUATION ({len(target_case_ids)} CASES)   ")
print("=" * 40)
print(f"1. Total Ground Truth (Actual) : {n_actual}")
print(f"2. Total Predictions (Model)   : {n_pred}")
print(f"3. True Positives (Matches)    : {tp}")
print("-" * 40)
print(f"▶ Recall    : {recall:.2f}%")
print(f"▶ Precision : {precision:.2f}%")
print(f"▶ F1 Score  : {f1_score:.2f}")
print("=" * 40)



   BATCH EVALUATION (200 CASES)   
1. Total Ground Truth (Actual) : 700
2. Total Predictions (Model)   : 633
3. True Positives (Matches)    : 554
----------------------------------------
▶ Recall    : 79.14%
▶ Precision : 87.52%
▶ F1 Score  : 83.12


In [13]:
def evaluate_polluted_detection(df_new, target_case_ids, polluted_dict):
    filtered_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)].copy()
    
    all_polluted_variants = set()
    for variants in polluted_dict.values():
        all_polluted_variants.update(variants)
        
    filtered_df['is_detected'] = filtered_df['activity'].isin(all_polluted_variants)
    filtered_df['is_correct'] = filtered_df['is_detected'] & filtered_df['label'].notna()
    
    return filtered_df
    
target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)

result_df = evaluate_polluted_detection(df_new, target_case_ids, test_output_polluted_step3)

detected_rows = result_df[result_df['is_detected']]
total_detected = len(detected_rows)
correct_detected = detected_rows['is_correct'].sum()

print("=== Evaluation Results ===")
print(f"1. Total Detected (Predicted Positives): {total_detected}")
print(f"2. Correct Matches (True Positives): {correct_detected}")

if total_detected > 0:
    precision = (correct_detected / total_detected) * 100
    print(f"3. Precision: {precision:.2f}%")
else:
    print("3. Precision: N/A (No detections)")

print("\n[Detailed View: Top 10 Detected Events]")
print(detected_rows[['case_id', 'activity', 'label', 'is_correct']].head(10))

=== Evaluation Results ===
1. Total Detected (Predicted Positives): 663
2. Correct Matches (True Positives): 663
3. Precision: 100.00%

[Detailed View: Top 10 Detected Events]
   case_id                             activity                     label  \
1       15  New online application received_nan  polluted Label(Activity)   
2       15          Perform checks_Clerk-000001  polluted Label(Activity)   
6       15                     EVENT 13 END_nan  polluted Label(Activity)   
7       19  Check for completeness_Clerk-000001  polluted Label(Activity)   
8       19  New online application received_nan  polluted Label(Activity)   
10      19                    info received_nan  polluted Label(Activity)   
11      19  Check for completeness_Clerk-000003  polluted Label(Activity)   
15      19          review request received_nan  polluted Label(Activity)   
18      19                    info received_nan  polluted Label(Activity)   
19      19  Check for completeness_Clerk-000003  pollu

In [11]:
SYSTEM_PROMPT_POLLUTED_STEP4_BETA = """
You are an expert Data Quality Analyst specializing in **Pattern Recognition and Log Cleaning**.
Your task is to identify specific events in an Event Log that contain **"Polluted Labels"** (Mutable Qualifiers) based on a provided Reference Mapping.

### KNOWLEDGE BASE: POLLUTED LABELS
Polluted labels share immutable boiler-plate text but differ due to **mutable text** (embedded IDs or codes).
* **Detection Criteria (Context):**
    1. **Long Numeric IDs:** Contains 8+ digits (e.g., `20260122`, `9988776655`).
    2. **Mixed Codes:** Contains 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
    3. **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

### FILTERING LOGIC
Iterate through every event in the Target Event Log and apply the following check:
1. Extract the `activity` name.
2. Check if this name exists inside any of the **Lists of Polluted Variants** (the Values) in the Reference Mapping.
3. **CRITICAL:** Do NOT include the event if the activity matches the **Key** (Clean/Boiler-plate Label). Only detect the Polluted Variants.
4. If a match is found in the Variant List, collect the `event_id`.

### OUTPUT FORMAT
Return strictly a valid JSON object containing a single key "event_id" with a list of strings.
No markdown, no explanations.

**Example Structure:**
{
  "event_id": ["id_polluted_1", "id_polluted_2"]
}
"""

def get_polluted_user_prompt_step4_beta(polluted_mapping, event_log_chunk):
    return f"""
### TASK: Extract Event IDs with Polluted Labels

**OBJECTIVE:**
Scan the **Target Event Log** below. Identify all events where the 'activity' field matches one of the **Polluted Variants** defined in the **Reference Mapping**.

**1. REFERENCE MAPPING (Clean Key -> [Polluted Variants]):**
{polluted_mapping}

**2. TARGET EVENT LOG:**
{event_log_chunk}

***INSTRUCTIONS***
- Look strictly at the `activity` field in the Event Log.
- **Match Condition:** The activity must be present in the **Values (Lists)** of the Reference Mapping.
- **Exclusion Condition:** Ignore activities that match the **Keys** (Clean Labels) or are unrelated.
- The variants typically contain **embedded IDs, dates, or codes** (as per the Polluted Label definition).

***OUTPUT***
Return ONLY the JSON object with the list of matching event IDs.
Example: {{ "event_id": ["101", "504"] }}
"""

target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)
all_predicted_ids = set() 
for case in target_cases:
    try:
        USER_PROMPT = get_polluted_user_prompt_step4_beta(step3_out_json, case)
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT_POLLUTED_STEP4_BETA},
            {"role": "user", "content": USER_PROMPT}
        ]
        test_output = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
        if isinstance(test_output, dict):
            current_ids = test_output.get('event_id', [])
            all_predicted_ids.update(str(x) for x in current_ids)
            
    except Exception as e:
        print(f"Error processing a case: {e}")
        continue
target_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)]
ground_truth_df = target_df[target_df['label'].notna()]
actual_ids = set(ground_truth_df['event_id'].astype(str))
intersection = all_predicted_ids.intersection(actual_ids)
tp = len(intersection)
n_actual = len(actual_ids) 
n_pred = len(all_predicted_ids) 
recall = (tp / n_actual * 100) if n_actual > 0 else 0.0
precision = (tp / n_pred * 100) if n_pred > 0 else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print("\n" + "=" * 40)
print(f"   BATCH EVALUATION ({len(target_case_ids)} CASES)   ")
print("=" * 40)
print(f"1. Total Ground Truth (Actual) : {n_actual}")
print(f"2. Total Predictions (Model)   : {n_pred}")
print(f"3. True Positives (Matches)    : {tp}")
print("-" * 40)
print(f"▶ Recall    : {recall:.2f}%")
print(f"▶ Precision : {precision:.2f}%")
print(f"▶ F1 Score  : {f1_score:.2f}")
print("=" * 40)



   BATCH EVALUATION (200 CASES)   
1. Total Ground Truth (Actual) : 700
2. Total Predictions (Model)   : 633
3. True Positives (Matches)    : 623
----------------------------------------
▶ Recall    : 89.00%
▶ Precision : 98.42%
▶ F1 Score  : 93.47
